# LAB 3 — End-to-End Multimodal Predictive Pipeline

Pipeline:
**Dataset Adapter → Multimodal Vector → Transformation/Fusion → User Profile → Pair Features → Predictive Model → Ranking → Evaluation**

Classification ใช้ทำนาย `like / not-like`; Regression ใช้ทำนาย rating และประเมิน MAE/RMSE

In [1]:
# ============================================================
# Part 0 — Colab Environment
# ============================================================
!pip -q install sentence-transformers transformers librosa soundfile gradio gensim

import os, math, warnings, json, random
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score, mean_absolute_error, mean_squared_error

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 32.1 MB/s eta 0:00:00


In [2]:
# ============================================================
# Part 1 — DATASET ADAPTER
# แก้ไข "เฉพาะส่วนนี้" เมื่อนำโปรแกรมไปใช้กับ Dataset อื่น
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/Colab Notebooks")
DATA_ROOT = BASE / "Dataset_Chapter3_4"

# Dataset_Chapter3_4 ที่สร้างไว้มี data/ เป็น subfolder
DATA_DIR = DATA_ROOT / "data" if (DATA_ROOT / "data").exists() else DATA_ROOT

# ---------- Dataset-specific file names ----------
ITEM_FILE = DATA_DIR / "metadata.csv"
RATING_FILE = DATA_DIR / "ratings.csv"

# ---------- Dataset-specific column aliases ----------
ITEM_ALIASES = {
    "movie_id":"item_id", "id":"item_id",
    "movie_title":"title", "name":"title",
    "genres":"genre", "description":"synopsis",
    "plot":"synopsis", "summary":"synopsis"
}
RATING_ALIASES = {
    "uid":"user_id", "user":"user_id",
    "iid":"item_id", "movie_id":"item_id",
    "score":"rating"
}

# ---------- Dataset-specific media folders ----------
MEDIA_DIRS = {
    "image": DATA_DIR / "images",
    "audio": DATA_DIR / "audio",
    "video": DATA_DIR / "video",
}
MEDIA_EXTS = {
    "image":[".png",".jpg",".jpeg",".webp"],
    "audio":[".wav",".mp3",".flac",".m4a"],
    "video":[".mp4",".mov",".avi",".mkv"],
}

def _rename_if_needed(df, aliases):
    df=df.copy()
    for old,new in aliases.items():
        if old in df.columns and new not in df.columns:
            df=df.rename(columns={old:new})
    return df

def _find_media(folder, item_id, exts):
    for ext in exts:
        p=folder/f"{item_id}{ext}"
        if p.exists():
            return str(p)
    return None

def load_dataset():
    items=_rename_if_needed(pd.read_csv(ITEM_FILE),ITEM_ALIASES)
    ratings=_rename_if_needed(pd.read_csv(RATING_FILE),RATING_ALIASES)

    need_i=["item_id","title","genre","synopsis"]
    need_r=["user_id","item_id","rating"]
    miss_i=[c for c in need_i if c not in items.columns]
    miss_r=[c for c in need_r if c not in ratings.columns]
    if miss_i or miss_r:
        raise ValueError(f"Missing item columns={miss_i}, rating columns={miss_r}")

    items["item_id"]=items["item_id"].astype(str)
    ratings["item_id"]=ratings["item_id"].astype(str)
    ratings["user_id"]=ratings["user_id"].astype(str)
    ratings["rating"]=pd.to_numeric(ratings["rating"])

    for c in ["title","genre","synopsis"]:
        items[c]=items[c].fillna("").astype(str)

    # Text input used by Chapter 3 / text modality
    items["text_content"]=(items["genre"]+" "+items["synopsis"]).str.strip()

    for modality in ["image","audio","video"]:
        items[f"{modality}_path"]=items["item_id"].apply(
            lambda x: _find_media(MEDIA_DIRS[modality],x,MEDIA_EXTS[modality])
        )

    return items,ratings

items,ratings=load_dataset()
ITEM_INDEX={iid:i for i,iid in enumerate(items["item_id"])}

print("DATA_ROOT =",DATA_ROOT)
print("Items =",len(items),"Users =",ratings["user_id"].nunique(),"Ratings =",len(ratings))
display(items[["item_id","title","genre","image_path","audio_path","video_path"]].head())

Mounted at /content/drive
DATA_ROOT = /content/drive/MyDrive/Colab Notebooks/Dataset_Chapter3_4
Items = 12 Users = 8 Ratings = 32


,item_id,title,genre,image_path,audio_path,video_path
0,i1,Toy Story 5,Animation|Family,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...
1,i2,Inside Out 2,Animation|Family,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...
2,i3,The Wild Robot,Animation|Sci-Fi,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...
3,i4,It Ends with Us,Drama|Romance,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...
4,i5,Wicked,Fantasy|Musical,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...,/content/drive/MyDrive/Colab Notebooks/Dataset...


In [3]:
# ============================================================
# Part 2 — REUSABLE CORE
# ส่วนนี้ไม่ผูกกับ MovieMini-12 โดยตรง
# ขอเพียง items / ratings อยู่ใน canonical schema
# ============================================================
LIKED_THRESHOLD=4.0
DISLIKED_THRESHOLD=2.0
N_USERS=max(1,ratings["user_id"].nunique())
POPULARITY=ratings.groupby("item_id")["user_id"].nunique().to_dict()

def l2_rows(X):
    return normalize(np.asarray(X,dtype=float))

def transform_vectors(X,mode="L2",pca_dim=16):
    X=np.asarray(X,dtype=float)
    if mode=="Raw":
        return X.copy()
    if mode=="L2":
        return normalize(X)
    if mode=="Center+L2":
        return normalize(X-X.mean(axis=0,keepdims=True))
    if mode=="Standard+L2":
        return normalize(StandardScaler().fit_transform(X))
    if mode=="PCA+L2":
        d=min(pca_dim, max(1,X.shape[0]-1), X.shape[1])
        return normalize(PCA(n_components=d,random_state=42).fit_transform(X))
    raise ValueError(mode)

def _unit(v):
    v=np.asarray(v,dtype=float)
    n=np.linalg.norm(v)
    return v if n==0 else v/n

# User Profile 1: Positive-only
def profile_positive_only(uid,X,rdf):
    r=rdf[rdf["user_id"].astype(str)==str(uid)]
    liked=r[r["rating"]>=LIKED_THRESHOLD]
    idx=[ITEM_INDEX[str(i)] for i in liked["item_id"] if str(i) in ITEM_INDEX]
    if not idx:
        idx=[ITEM_INDEX[str(i)] for i in r["item_id"] if str(i) in ITEM_INDEX]
    return _unit(X[idx].mean(axis=0)) if idx else np.zeros(X.shape[1])

# User Profile 2: Rating-centered
def profile_rating_centered(uid,X,rdf):
    r=rdf[rdf["user_id"].astype(str)==str(uid)].copy()
    r=r[r["item_id"].astype(str).isin(ITEM_INDEX)]
    if r.empty: return np.zeros(X.shape[1])
    idx=np.array([ITEM_INDEX[str(i)] for i in r["item_id"]])
    w=r["rating"].astype(float).values-float(r["rating"].mean())
    den=np.abs(w).sum()
    p=X[idx].mean(axis=0) if den<=1e-12 else (X[idx]*w[:,None]).sum(axis=0)/den
    return _unit(p)

# User Profile 3: Rocchio
def profile_rocchio(uid,X,rdf,alpha=1.0,beta=1.0):
    r=rdf[rdf["user_id"].astype(str)==str(uid)]
    pos=r[r["rating"]>=LIKED_THRESHOLD]
    neg=r[r["rating"]<=DISLIKED_THRESHOLD]
    pi=[ITEM_INDEX[str(i)] for i in pos["item_id"] if str(i) in ITEM_INDEX]
    ni=[ITEM_INDEX[str(i)] for i in neg["item_id"] if str(i) in ITEM_INDEX]
    vp=X[pi].mean(axis=0) if pi else np.zeros(X.shape[1])
    vn=X[ni].mean(axis=0) if ni else np.zeros(X.shape[1])
    p=alpha*vp-beta*vn
    if np.linalg.norm(p)==0:
        hi=[ITEM_INDEX[str(i)] for i in r["item_id"] if str(i) in ITEM_INDEX]
        p=X[hi].mean(axis=0) if hi else p
    return _unit(p)

PROFILE_METHODS=["Positive-only","Rating-centered","Rocchio"]

def make_profile(uid,X,rdf,method="Rating-centered",alpha=1.0,beta=1.0):
    if method=="Positive-only": return profile_positive_only(uid,X,rdf)
    if method=="Rating-centered": return profile_rating_centered(uid,X,rdf)
    if method=="Rocchio": return profile_rocchio(uid,X,rdf,alpha,beta)
    raise ValueError(method)

def rank_by_vector(uid,X,rdf,method="Rating-centered",candidates=None,alpha=1.0,beta=1.0):
    p=make_profile(uid,X,rdf,method,alpha,beta)
    s=cosine_similarity(p.reshape(1,-1),X).ravel()
    seen=set(rdf.loc[rdf["user_id"].astype(str)==str(uid),"item_id"].astype(str))
    cand=candidates if candidates is not None else [iid for iid in items["item_id"] if iid not in seen]
    ranked=sorted(cand,key=lambda iid:s[ITEM_INDEX[iid]],reverse=True)
    return ranked,s,p

# ---------------- Accuracy / Ranking metrics ----------------
def precision_at_k(recs,rel,k): return len(set(recs[:k])&set(rel))/k if k else 0.0
def recall_at_k(recs,rel,k):
    rel=set(rel); return len(set(recs[:k])&rel)/len(rel) if rel else 0.0
def f1_at_k(recs,rel,k):
    p=precision_at_k(recs,rel,k); r=recall_at_k(recs,rel,k)
    return 0.0 if p+r==0 else 2*p*r/(p+r)
def hr_at_k(recs,rel,k): return float(bool(set(recs[:k])&set(rel)))
def reciprocal_rank(recs,rel):
    rel=set(rel)
    for i,x in enumerate(recs,1):
        if x in rel:return 1/i
    return 0.0
def ap_at_k(recs,rel,k):
    rel=set(rel); hit=0;s=0
    for i,x in enumerate(recs[:k],1):
        if x in rel: hit+=1;s+=hit/i
    return s/min(len(rel),k) if rel else 0.0
def ndcg_at_k(recs,rel,k):
    rel=set(rel)
    dcg=sum(1/math.log2(i+2) for i,x in enumerate(recs[:k]) if x in rel)
    idcg=sum(1/math.log2(i+2) for i in range(min(len(rel),k)))
    return dcg/idcg if idcg else 0.0

# ---------------- Beyond-Accuracy metrics ----------------
def ild(recs,X):
    idx=[ITEM_INDEX[i] for i in recs if i in ITEM_INDEX]
    if len(idx)<2:return 0.0
    D=1-cosine_similarity(X[idx])
    v=D[np.triu_indices_from(D,k=1)]
    return float(v.mean()) if len(v) else 0.0

def novelty(recs):
    vals=[-math.log2(max(POPULARITY.get(i,1)/N_USERS,1e-12)) for i in recs]
    return float(np.mean(vals)) if vals else 0.0

def calibration_genre(uid,recs,rdf):
    hist=rdf[rdf["user_id"].astype(str)==str(uid)]
    def gs(ids):
        out=[]
        for iid in ids:
            if iid in ITEM_INDEX:
                out += [x.strip() for x in str(items.iloc[ITEM_INDEX[iid]]["genre"]).replace("|",",").split(",") if x.strip()]
        return out
    a=gs(hist["item_id"].astype(str).tolist()); b=gs(recs)
    if not a or not b:return np.nan
    cats=sorted(set(a)|set(b))
    pa=np.array([a.count(c) for c in cats],float); pa/=pa.sum()
    pb=np.array([b.count(c) for c in cats],float); pb/=pb.sum()
    return float(1-0.5*np.abs(pa-pb).sum())

def coverage(all_recs): return len(set(all_recs))/len(items) if len(items) else 0.0

def gini(all_recs):
    c=pd.Series(all_recs).value_counts()
    x=np.sort(np.array([c.get(i,0) for i in items["item_id"]],float))
    if x.sum()==0:return 0.0
    n=len(x)
    return float(2*np.sum(np.arange(1,n+1)*x)/(n*x.sum())-(n+1)/n)

# ---------------- Offline splits ----------------
def split_loo(rdf):
    folds=[]
    for uid,g in rdf.groupby("user_id"):
        p=g[g["rating"]>=LIKED_THRESHOLD]
        if p.empty: continue
        hold=p.sort_values("rating").iloc[-1]
        folds.append((str(uid),rdf.drop(index=hold.name),str(hold["item_id"])))
    return folds

def split_random_holdout(rdf,seed=42):
    rng=np.random.default_rng(seed); folds=[]
    for uid,g in rdf.groupby("user_id"):
        p=g[g["rating"]>=LIKED_THRESHOLD]
        if p.empty:continue
        hold=p.iloc[int(rng.integers(0,len(p)))]
        folds.append((str(uid),rdf.drop(index=hold.name),str(hold["item_id"])))
    return folds

def split_temporal(rdf):
    if "timestamp" not in rdf.columns:return None
    folds=[]
    for uid,g in rdf.groupby("user_id"):
        p=g[g["rating"]>=LIKED_THRESHOLD].sort_values("timestamp")
        if p.empty:continue
        hold=p.iloc[-1]
        folds.append((str(uid),rdf.drop(index=hold.name),str(hold["item_id"])))
    return folds

def evaluate_vector_space(X,label="",profile_method="Rating-centered",k=3,folds=None,
                          candidate_mode="Full Catalog",n_neg=20,alpha=1.0,beta=1.0,seed=42):
    folds=folds or split_loo(ratings)
    rng=np.random.default_rng(seed)
    rows=[];all_recs=[]
    for uid,train,heldout in folds:
        seen=set(train.loc[train["user_id"].astype(str)==uid,"item_id"].astype(str))
        catalog=[i for i in items["item_id"] if i not in seen]
        if heldout not in catalog: catalog.append(heldout)
        if candidate_mode=="Sampled Negatives":
            neg=[i for i in catalog if i!=heldout]
            take=min(n_neg,len(neg))
            cand=[heldout]+(list(rng.choice(neg,take,replace=False)) if take else [])
        else:
            cand=catalog
        ranked,s,_=rank_by_vector(uid,X,train,profile_method,cand,alpha,beta)
        top=ranked[:k];all_recs+=top
        y=np.array([1 if i==heldout else 0 for i in ranked])
        ys=np.array([s[ITEM_INDEX[i]] for i in ranked])
        auc=roc_auc_score(y,ys) if len(np.unique(y))==2 else np.nan
        rows.append({
            "user_id":uid,"heldout":heldout,
            f"Precision@{k}":precision_at_k(top,[heldout],k),
            f"Recall@{k}":recall_at_k(top,[heldout],k),
            f"F1@{k}":f1_at_k(top,[heldout],k),
            f"HR@{k}":hr_at_k(top,[heldout],k),
            "MRR":reciprocal_rank(ranked,[heldout]),
            f"MAP@{k}":ap_at_k(top,[heldout],k),
            f"nDCG@{k}":ndcg_at_k(top,[heldout],k),
            "AUC":auc,
            "ILD":ild(top,X),"Novelty":novelty(top),
            "Calibration":calibration_genre(uid,top,train),
        })
    d=pd.DataFrame(rows)
    s=d.select_dtypes(include=np.number).mean().to_dict() if len(d) else {}
    s["Coverage"]=coverage(all_recs);s["Gini"]=gini(all_recs);s["Label"]=label
    return pd.Series(s),d

def quick_eval(X,label):
    rows=[]
    for p in PROFILE_METHODS:
        s,_=evaluate_vector_space(X,label,p,k=3)
        rows.append({"Vector":label,"Profile":p,**s.to_dict()})
    cols=["Vector","Profile","HR@3","MRR","nDCG@3","Coverage","Gini","ILD","Novelty","Calibration"]
    return pd.DataFrame(rows)[[c for c in cols if c in pd.DataFrame(rows).columns]]

In [4]:
# ============================================================
# Part 3 — MULTIMODAL ENCODERS
# สอดคล้องกับแนวคิดในสไลด์: Text/Image/Audio/Video,
# CNN, Transformer, pooling และ L2 normalization
# ============================================================
import torch, torch.nn as nn
from PIL import Image
import cv2, librosa
from torchvision.models import resnet18, ResNet18_Weights, vit_b_16, ViT_B_16_Weights
from sentence_transformers import SentenceTransformer
from transformers import AutoProcessor, AutoModel

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE=DATA_ROOT/"vector_cache"
CACHE.mkdir(parents=True,exist_ok=True)

def cache_vec(name,builder,rebuild=False):
    p=CACHE/f"{name}.npy"
    if p.exists() and not rebuild:
        print("cache:",name);return np.load(p)
    X=np.asarray(builder(),dtype=np.float32)
    np.save(p,X);print("saved:",name,X.shape);return X

# ---------- Text: TF-IDF ----------
def build_tfidf():
    v=TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=1)
    X=v.fit_transform(items["text_content"]).toarray()
    return X,v

X_text_tfidf,TFIDF_VECTORIZER=build_tfidf()

# ---------- Text Transformer: SBERT ----------
def _sbert():
    m=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",device=str(DEVICE))
    return m.encode(items["text_content"].tolist(),show_progress_bar=True,normalize_embeddings=True)
X_text_sbert=cache_vec("text_sbert",_sbert)

# ---------- Text CNN: teaching demonstration ----------
# Train a small TextCNN to predict primary genre, then use pooled CNN features as item vectors.
# Dataset เล็กมาก: ใช้เพื่อให้เห็นกลไก CNN จาก token -> convolution -> max pooling เท่านั้น
def build_textcnn_vectors(epochs=30,emb_dim=48,filters=32):
    texts=[str(x).lower().split() for x in items["text_content"]]
    vocab={"<PAD>":0,"<UNK>":1}
    for t in texts:
        for w in t:
            if w not in vocab:vocab[w]=len(vocab)
    maxlen=max(4,min(80,max(len(t) for t in texts)))
    seq=np.zeros((len(texts),maxlen),dtype=np.int64)
    for i,t in enumerate(texts):
        ids=[vocab.get(w,1) for w in t][:maxlen]
        seq[i,:len(ids)]=ids
    labels=items["genre"].str.split("[,|]",regex=True).str[0].fillna("Unknown")
    cats={g:i for i,g in enumerate(sorted(labels.unique()))}
    y=np.array([cats[g] for g in labels],dtype=np.int64)

    class TextCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.emb=nn.Embedding(len(vocab),emb_dim,padding_idx=0)
            self.convs=nn.ModuleList([nn.Conv1d(emb_dim,filters,k) for k in [2,3,4]])
            self.fc=nn.Linear(filters*3,max(1,len(cats)))
        def features(self,x):
            e=self.emb(x).transpose(1,2)
            fs=[torch.relu(c(e)).max(dim=2).values for c in self.convs]
            return torch.cat(fs,dim=1)
        def forward(self,x):return self.fc(self.features(x))

    model=TextCNN().to(DEVICE)
    if len(cats)>1:
        opt=torch.optim.Adam(model.parameters(),lr=0.01)
        X=torch.tensor(seq,dtype=torch.long,device=DEVICE)
        Y=torch.tensor(y,dtype=torch.long,device=DEVICE)
        for _ in range(epochs):
            opt.zero_grad();loss=nn.CrossEntropyLoss()(model(X),Y);loss.backward();opt.step()
    model.eval()
    with torch.no_grad():
        z=model.features(torch.tensor(seq,dtype=torch.long,device=DEVICE)).cpu().numpy()
    return normalize(z)
X_text_cnn=cache_vec("textcnn_demo",build_textcnn_vectors)

# ---------- Shared CNN / ViT for Image and Video ----------
RW=ResNet18_Weights.DEFAULT
RESNET_PRE=RW.transforms()
RESNET=resnet18(weights=RW);RESNET.fc=nn.Identity();RESNET=RESNET.eval().to(DEVICE)

VW=ViT_B_16_Weights.DEFAULT
VIT_PRE=VW.transforms()
VIT=vit_b_16(weights=VW);VIT.heads=nn.Identity();VIT=VIT.eval().to(DEVICE)

def encode_pil(images,model,prep,batch=8):
    rows=[]
    for i in range(0,len(images),batch):
        x=torch.stack([prep(im.convert("RGB")) for im in images[i:i+batch]]).to(DEVICE)
        with torch.no_grad():rows.append(model(x).cpu().numpy())
    return np.vstack(rows)

# ---------- Image CNN / Transformer ----------
def _image_cnn():
    ims=[Image.open(p) if p else Image.new("RGB",(224,224)) for p in items["image_path"]]
    return normalize(encode_pil(ims,RESNET,RESNET_PRE))
def _image_vit():
    ims=[Image.open(p) if p else Image.new("RGB",(224,224)) for p in items["image_path"]]
    return normalize(encode_pil(ims,VIT,VIT_PRE))
X_image_cnn=cache_vec("image_resnet18",_image_cnn)
X_image_vit=cache_vec("image_vit",_image_vit)

# ---------- Audio CNN: Log-Mel spectrogram -> CNN ----------
def audio_to_image(path,sr=16000):
    if not path:return Image.new("RGB",(224,224))
    y,_=librosa.load(path,sr=sr,mono=True)
    mel=librosa.feature.melspectrogram(y=y,sr=sr,n_mels=128,n_fft=1024,hop_length=256)
    db=librosa.power_to_db(mel,ref=np.max)
    lo,hi=float(db.min()),float(db.max())
    x=(db-lo)/(hi-lo) if hi>lo else np.zeros_like(db)
    return Image.fromarray(np.uint8(np.clip(x*255,0,255))).convert("RGB")
def _audio_cnn():
    ims=[audio_to_image(p) for p in items["audio_path"]]
    return normalize(encode_pil(ims,RESNET,RESNET_PRE))
X_audio_cnn=cache_vec("audio_logmel_resnet18",_audio_cnn)

# ---------- Audio Transformer: Wav2Vec2 ----------
def _audio_w2v():
    proc=AutoProcessor.from_pretrained("facebook/wav2vec2-base-960h")
    mod=AutoModel.from_pretrained("facebook/wav2vec2-base-960h").eval().to(DEVICE)
    rows=[]
    for p in items["audio_path"]:
        if not p:
            rows.append(np.zeros(768));continue
        y,_=librosa.load(p,sr=16000,mono=True);y=y[:16000*12]
        inp=proc(y,sampling_rate=16000,return_tensors="pt")
        with torch.no_grad():
            z=mod(inp["input_values"].to(DEVICE)).last_hidden_state.mean(1).squeeze(0).cpu().numpy()
        rows.append(z)
    return normalize(np.vstack(rows))
X_audio_transformer=cache_vec("audio_wav2vec2",_audio_w2v)

# ---------- Video: frame sampling -> CNN/ViT -> temporal mean pooling ----------
def frames(path,n=8):
    if not path:return []
    cap=cv2.VideoCapture(str(path)); total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if total<=0:cap.release();return []
    ids=np.linspace(0,total-1,min(n,total)).astype(int);out=[]
    for j in ids:
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(j));ok,f=cap.read()
        if ok:out.append(Image.fromarray(cv2.cvtColor(f,cv2.COLOR_BGR2RGB)))
    cap.release();return out

def _video(model,prep,dim):
    rows=[]
    for p in items["video_path"]:
        fs=frames(p)
        rows.append(encode_pil(fs,model,prep).mean(0) if fs else np.zeros(dim))
    return normalize(np.vstack(rows))
X_video_cnn=cache_vec("video_resnet_mean",lambda:_video(RESNET,RESNET_PRE,512))
X_video_vit=cache_vec("video_vit_mean",lambda:_video(VIT,VIT_PRE,768))

VECTOR_LIBRARY={
    "Text-TFIDF":X_text_tfidf,
    "Text-CNN":X_text_cnn,
    "Text-SBERT":X_text_sbert,
    "Image-CNN":X_image_cnn,
    "Image-ViT":X_image_vit,
    "Audio-CNN":X_audio_cnn,
    "Audio-Wav2Vec2":X_audio_transformer,
    "Video-CNN":X_video_cnn,
    "Video-ViT":X_video_vit,
}
print({k:v.shape for k,v in VECTOR_LIBRARY.items()})

cache: text_sbert
cache: textcnn_demo
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 66.4MB/s]


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:04<00:00, 69.9MB/s]


cache: image_resnet18
cache: image_vit
cache: audio_logmel_resnet18
cache: audio_wav2vec2
cache: video_resnet_mean
saved: video_vit_mean (12, 768)
{'Text-TFIDF': (12, 800), 'Text-CNN': (12, 96), 'Text-SBERT': (12, 384), 'Image-CNN': (12, 512), 'Image-ViT': (12, 768), 'Audio-CNN': (12, 512), 'Audio-Wav2Vec2': (12, 768), 'Video-CNN': (12, 512), 'Video-ViT': (12, 768)}


In [5]:
# ============================================================
# Part 4 — Fusion
# ============================================================
def early_fusion(vector_dict,weights=None):
    names=list(vector_dict)
    weights=weights or {n:1.0 for n in names}
    blocks=[]
    for n in names:
        X=normalize(vector_dict[n])
        blocks.append(np.sqrt(float(weights.get(n,1.0)))*X)
    return normalize(np.hstack(blocks))

DEFAULT_MODALITIES={
    "text":X_text_sbert,
    "image":X_image_cnn,
    "audio":X_audio_transformer,
    "video":X_video_vit
}
X_fused=early_fusion(DEFAULT_MODALITIES)

def late_fusion_rank(uid,vector_dict,rdf,profile_method="Rating-centered",
                     weights=None,candidates=None,alpha=1,beta=1):
    weights=weights or {n:1/len(vector_dict) for n in vector_dict}
    seen=set(rdf.loc[rdf["user_id"].astype(str)==str(uid),"item_id"].astype(str))
    cand=candidates if candidates is not None else [i for i in items["item_id"] if i not in seen]
    total={iid:0.0 for iid in cand}
    for name,X in vector_dict.items():
        ranked,s,_=rank_by_vector(uid,X,rdf,profile_method,cand,alpha,beta)
        for iid in cand: total[iid]+=weights.get(name,0)*float(s[ITEM_INDEX[iid]])
    ranked=sorted(cand,key=lambda x:total[x],reverse=True)
    return ranked,total

def evaluate_late_fusion(vector_dict,label="Late Fusion",profile_method="Rating-centered",
                         k=3,folds=None,weights=None,candidate_mode="Full Catalog",n_neg=20,seed=42):
    folds=folds or split_loo(ratings);rng=np.random.default_rng(seed)
    rows=[];all_recs=[]
    # ILD ใช้ early-fused semantic space เพื่อวัดความหลากหลายร่วม
    Xdiv=early_fusion(vector_dict)
    for uid,train,heldout in folds:
        seen=set(train.loc[train["user_id"].astype(str)==uid,"item_id"].astype(str))
        catalog=[i for i in items["item_id"] if i not in seen]
        if heldout not in catalog:catalog.append(heldout)
        if candidate_mode=="Sampled Negatives":
            neg=[i for i in catalog if i!=heldout]
            take=min(n_neg,len(neg));cand=[heldout]+(list(rng.choice(neg,take,replace=False)) if take else [])
        else:cand=catalog
        ranked,sc=late_fusion_rank(uid,vector_dict,train,profile_method,weights,cand)
        top=ranked[:k];all_recs+=top
        y=np.array([1 if i==heldout else 0 for i in ranked])
        ys=np.array([sc[i] for i in ranked])
        rows.append({
            f"Precision@{k}":precision_at_k(top,[heldout],k),
            f"Recall@{k}":recall_at_k(top,[heldout],k),
            f"F1@{k}":f1_at_k(top,[heldout],k),
            f"HR@{k}":hr_at_k(top,[heldout],k),
            "MRR":reciprocal_rank(ranked,[heldout]),
            f"MAP@{k}":ap_at_k(top,[heldout],k),
            f"nDCG@{k}":ndcg_at_k(top,[heldout],k),
            "AUC":roc_auc_score(y,ys) if len(np.unique(y))==2 else np.nan,
            "ILD":ild(top,Xdiv),"Novelty":novelty(top),
            "Calibration":calibration_genre(uid,top,train)
        })
    d=pd.DataFrame(rows);s=d.mean(numeric_only=True).to_dict()
    s["Coverage"]=coverage(all_recs);s["Gini"]=gini(all_recs);s["Label"]=label
    return pd.Series(s),d

## Part 5 — เลือก Multimodal Representation

In [6]:
# Transformer-rich / hybrid representation
X_BASE=early_fusion({
    "text":X_text_sbert,
    "image":X_image_vit,
    "audio":X_audio_transformer,
    "video":X_video_vit
})
X_MODEL=transform_vectors(X_BASE,"L2")
print(X_MODEL.shape)

(12, 2688)


## Part 6 — Interaction Train/Test Split

สร้าง profile และ pair features จาก train history เท่านั้น เพื่อลด data leakage

In [7]:
rng=np.random.default_rng(42)
tr=[];te=[]
for uid,g in ratings.groupby("user_id"):
    ids=g.index.to_numpy()
    if len(ids)<2:tr+=list(ids);continue
    n=max(1,int(round(len(ids)*0.25)))
    held=set(rng.choice(ids,size=n,replace=False))
    te+=list(held);tr += [i for i in ids if i not in held]
train_r=ratings.loc[tr].copy();test_r=ratings.loc[te].copy()
print("train",len(train_r),"test",len(test_r))

train 24 test 8


## Part 7 — User–Item Pair Features

Features:
- cosine(profile,item)
- absolute difference
- elementwise product

In [8]:
PROFILE_FOR_MODEL="Rating-centered"

def pair_feature(uid,iid,X,rdf,profile_method):
    p=make_profile(uid,X,rdf,profile_method)
    v=X[ITEM_INDEX[str(iid)]]
    cos=float(np.dot(p,v)/(max(np.linalg.norm(p)*np.linalg.norm(v),1e-12)))
    return np.concatenate([[cos],np.abs(p-v),p*v])

def make_pairs(df,X,history,profile_method):
    F=[];yc=[];yr=[];meta=[]
    for _,r in df.iterrows():
        iid=str(r["item_id"]);uid=str(r["user_id"])
        if iid not in ITEM_INDEX:continue
        F.append(pair_feature(uid,iid,X,history,profile_method))
        yr.append(float(r["rating"]));yc.append(int(float(r["rating"])>=4));meta.append((uid,iid))
    return np.asarray(F),np.asarray(yc),np.asarray(yr),meta

Ftr,yctr,yrtr,_=make_pairs(train_r,X_MODEL,train_r,PROFILE_FOR_MODEL)
Fte,ycte,yrte,meta_te=make_pairs(test_r,X_MODEL,train_r,PROFILE_FOR_MODEL)
print(Ftr.shape,Fte.shape)

(24, 5377) (8, 5377)


## Part 8 — Predictive Models + Pointwise Accuracy

In [9]:
from sklearn.linear_model import LogisticRegression,Ridge
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

CLF={
    "LogisticRegression":LogisticRegression(max_iter=2000,class_weight="balanced"),
    "RandomForestClassifier":RandomForestClassifier(n_estimators=200,random_state=42,class_weight="balanced")
}
REG={
    "Ridge":Ridge(alpha=1.0),
    "RandomForestRegressor":RandomForestRegressor(n_estimators=200,random_state=42)
}

cr=[]
for name,m in CLF.items():
    m.fit(Ftr,yctr);pr=m.predict_proba(Fte)[:,1];y=(pr>=.5).astype(int)
    cr.append({"Model":name,"Accuracy":accuracy_score(ycte,y),
               "Precision":precision_score(ycte,y,zero_division=0),
               "Recall":recall_score(ycte,y,zero_division=0),
               "F1":f1_score(ycte,y,zero_division=0),
               "AUC":roc_auc_score(ycte,pr) if len(np.unique(ycte))==2 else np.nan})
display(pd.DataFrame(cr).round(4))

rr=[]
for name,m in REG.items():
    m.fit(Ftr,yrtr);pr=m.predict(Fte)
    rr.append({"Model":name,"MAE":mean_absolute_error(yrte,pr),
               "RMSE":mean_squared_error(yrte,pr)**0.5})
display(pd.DataFrame(rr).round(4))

,Model,Accuracy,Precision,Recall,F1,AUC
0,LogisticRegression,0.625,0.625,1.0,0.7692,0.8000
1,RandomForestClassifier,0.625,0.625,1.0,0.7692,0.7333


,Model,MAE,RMSE
0,Ridge,1.0671,1.2352
1,RandomForestRegressor,0.9681,1.1480


## Part 9 — Predictive Ranking + Top-K / Beyond-Accuracy

นำ probability จาก classifier มาเป็น ranking score แล้ววัด metrics บทที่ 2

In [10]:
best=CLF["LogisticRegression"]

def predictive_rank(uid,model,X,history,profile_method):
    seen=set(history.loc[history["user_id"].astype(str)==str(uid),"item_id"].astype(str))
    cand=[i for i in items["item_id"] if i not in seen]
    if not cand:return [],{}
    F=np.vstack([pair_feature(uid,i,X,history,profile_method) for i in cand])
    pr=model.predict_proba(F)[:,1]
    sc=dict(zip(cand,pr))
    return sorted(cand,key=lambda i:sc[i],reverse=True),sc

rows=[];all_recs=[]
for uid,g in test_r.groupby("user_id"):
    rel=set(g.loc[g["rating"]>=4,"item_id"].astype(str))
    if not rel:continue
    ranked,sc=predictive_rank(uid,best,X_MODEL,train_r,PROFILE_FOR_MODEL)
    top=ranked[:3];all_recs+=top
    rows.append({
        "user_id":uid,"Precision@3":precision_at_k(top,rel,3),
        "Recall@3":recall_at_k(top,rel,3),"F1@3":f1_at_k(top,rel,3),
        "HR@3":hr_at_k(top,rel,3),"MRR":reciprocal_rank(ranked,rel),
        "MAP@3":ap_at_k(top,rel,3),"nDCG@3":ndcg_at_k(top,rel,3),
        "ILD":ild(top,X_MODEL),"Novelty":novelty(top),
        "Calibration":calibration_genre(uid,top,train_r)
    })
rank_detail=pd.DataFrame(rows)
summary=rank_detail.mean(numeric_only=True).to_dict()
summary["Coverage"]=coverage(all_recs);summary["Gini"]=gini(all_recs)
display(rank_detail.round(4));display(pd.Series(summary).to_frame("mean").round(4))

,user_id,Precision@3,Recall@3,F1@3,HR@3,MRR,MAP@3,nDCG@3,ILD,Novelty,Calibration
0,u1,0.3333,1.0,0.5,1.0,1.00,1.0,1.0000,0.4906,1.2767,0.4167
1,u3,0.3333,1.0,0.5,1.0,1.00,1.0,1.0000,0.4753,1.8050,0.5000
2,u4,0.3333,1.0,0.5,1.0,0.50,0.5,0.6309,0.3972,1.6667,0.3333
3,u5,0.0000,0.0,0.0,0.0,0.25,0.0,0.0000,0.3195,1.4150,0.1667
4,u7,0.3333,1.0,0.5,1.0,1.00,1.0,1.0000,0.5567,1.8050,0.3333


,mean
Precision@3,0.2667
Recall@3,0.8000
F1@3,0.4000
HR@3,0.8000
MRR,0.7500
MAP@3,0.7000
nDCG@3,0.7262
ILD,0.4479
Novelty,1.5937
Calibration,0.3500


## Part 10 — Experiment: User Profile × Transformation × Predictive Model

In [11]:
from sklearn.base import clone
exp=[]
for trans in ["L2","Center+L2"]:
    X=transform_vectors(X_BASE,trans)
    for prof in PROFILE_METHODS:
        A,ya,ra,_=make_pairs(train_r,X,train_r,prof)
        B,yb,rb,_=make_pairs(test_r,X,train_r,prof)
        for name,base in CLF.items():
            m=clone(base);m.fit(A,ya);pp=m.predict_proba(B)[:,1]
            exp.append({"Transformation":trans,"Profile":prof,"Model":name,
                        "AUC":roc_auc_score(yb,pp) if len(np.unique(yb))==2 else np.nan})
display(pd.DataFrame(exp).round(4))

,Transformation,Profile,Model,AUC
0,L2,Positive-only,LogisticRegression,0.8000
1,L2,Positive-only,RandomForestClassifier,0.9000
2,L2,Rating-centered,LogisticRegression,0.8000
3,L2,Rating-centered,RandomForestClassifier,0.7333
4,L2,Rocchio,LogisticRegression,0.0000
5,L2,Rocchio,RandomForestClassifier,0.6667
6,Center+L2,Positive-only,LogisticRegression,1.0000
7,Center+L2,Positive-only,RandomForestClassifier,0.9333
8,Center+L2,Rating-centered,LogisticRegression,0.9333
9,Center+L2,Rating-centered,RandomForestClassifier,1.0000


## Part 11 — GUI: Predictive Multimodal Recommendation

In [12]:
import gradio as gr

def gui_pred(uid,k):
    ranked,sc=predictive_rank(uid,best,X_MODEL,train_r,PROFILE_FOR_MODEL)
    top=ranked[:int(k)]
    hist=train_r[train_r["user_id"].astype(str)==str(uid)].merge(
        items[["item_id","title","genre"]],on="item_id",how="left"
    )[["title","genre","rating"]]
    rec=[]
    for j,iid in enumerate(top,1):
        r=items.iloc[ITEM_INDEX[iid]]
        rec.append({"Rank":j,"Title":r["title"],"Genre":r["genre"],
                    "P(like)":round(float(sc[iid]),4),"Synopsis":r["synopsis"]})
    first=items.iloc[ITEM_INDEX[top[0]]] if top else None
    return hist,pd.DataFrame(rec),(
        first["image_path"] if first is not None else None
    ),(first["audio_path"] if first is not None else None),(first["video_path"] if first is not None else None)

demo=gr.Interface(
    gui_pred,
    [gr.Dropdown(sorted(ratings["user_id"].unique()),label="User"),gr.Slider(1,5,3,1,label="Top-K")],
    [gr.Dataframe(label="Train History"),gr.Dataframe(label="Recommendations"),
     gr.Image(label="Top-1 Image"),gr.Audio(label="Top-1 Audio"),gr.Video(label="Top-1 Video")],
    title="Predictive Multimodal Recommender"
)
demo.launch(debug=False)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://243aae3790fe5b6be1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
